# Exp0.1 — Проверка гипотезы Adaptive Refinement

## Суть задачи

Есть 2D числовое поле (матрица) — эталон **GT**, созданный синтетически:
- гладкие пятна (гауссианы),
- резкие границы (жёсткие фигуры),
- пустые области.

**Процесс:**
1. Строим грубую аппроксимацию `coarse(GT)` — downsample + bilinear upsample.
2. Есть «дорогая» операция `refine(tile)` — применяется к отдельным тайлам.
3. Бюджет ограничен: нельзя refine всё.

**Сравниваем два метода:**
- **Baseline**: выбирает K тайлов случайно.
- **Adaptive**: выбирает K тайлов с наибольшей локальной ошибкой (rho = MSE тайла).

**Вопрос:** При одинаковом бюджете даёт ли adaptive лучшее качество?

## Критерий успеха
- При одинаковом бюджете средний MSE (особенно MSE_edge) меньше у adaptive.
- Устойчиво на большинстве coverage и сцен.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import zoom, sobel
from collections import OrderedDict
import warnings, time

warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════
# КОНФИГ
# ═══════════════════════════════════════════
H = 512
W = 512
tile_size = 32
downsample_factor = 4

alpha = 0.25
n_iter = 8

coverage_list = [0.01, 0.02, 0.05, 0.1, 0.2]

n_seeds_debug = 3
n_seeds_final = 30

mode = "debug"  # "debug" или "final"

scene_modes = ["A_sparse_edges", "B_many_edges", "C_smooth_only"]

edge_quantile = 0.9
edge_eps = 1e-6

max_static_images = 10

save_images = False  # ставить True для массового сохранения

# ─── Derived ───
n_seeds = n_seeds_debug if mode == "debug" else n_seeds_final
tiles_y = H // tile_size
tiles_x = W // tile_size
n_tiles_total = tiles_y * tiles_x

print(f"Mode: {mode}, seeds: {n_seeds}")
print(f"Grid: {tiles_y}x{tiles_x} = {n_tiles_total} tiles")
print(f"Coverage list: {coverage_list}")


In [ ]:
def _add_gaussian(field, cy, cx, sigma, amplitude, H, W):
    yy, xx = np.mgrid[0:H, 0:W]
    g = amplitude * np.exp(-((yy - cy)**2 + (xx - cx)**2) / (2 * sigma**2))
    field += g

def _add_rect(field, y0, x0, y1, x1, value):
    field[y0:y1, x0:x1] = value

def _add_circle(field, cy, cx, r, value, H, W):
    yy, xx = np.mgrid[0:H, 0:W]
    mask = (yy - cy)**2 + (xx - cx)**2 <= r**2
    field[mask] = value

def generate_gt(scene, seed, H, W):
    rng = np.random.RandomState(seed)
    field = np.zeros((H, W), dtype=np.float32)

    if scene == "A_sparse_edges":
        n_gauss = rng.randint(3, 7)
        for _ in range(n_gauss):
            cy, cx = rng.randint(50, H-50), rng.randint(50, W-50)
            sigma = rng.uniform(30, 80)
            amp = rng.uniform(0.3, 1.0)
            _add_gaussian(field, cy, cx, sigma, amp, H, W)
        # 1-2 hard shapes
        _add_rect(field,
                  rng.randint(50, H//2), rng.randint(50, W//2),
                  rng.randint(H//2, H-50), rng.randint(W//2, W-50),
                  rng.uniform(0.5, 1.0))
        _add_circle(field,
                    rng.randint(100, H-100), rng.randint(100, W-100),
                    rng.randint(30, 70), rng.uniform(0.6, 1.0), H, W)

    elif scene == "B_many_edges":
        n_rects = rng.randint(15, 30)
        for _ in range(n_rects):
            h_ = rng.randint(10, 60)
            w_ = rng.randint(10, 60)
            y0 = rng.randint(0, H - h_)
            x0 = rng.randint(0, W - w_)
            _add_rect(field, y0, x0, y0 + h_, x0 + w_, rng.uniform(0.3, 1.0))
        # a few thin stripes
        for _ in range(rng.randint(3, 8)):
            y0 = rng.randint(0, H - 5)
            _add_rect(field, y0, 0, y0 + rng.randint(2, 6), W, rng.uniform(0.4, 0.9))

    elif scene == "C_smooth_only":
        n_gauss = rng.randint(5, 12)
        for _ in range(n_gauss):
            cy, cx = rng.randint(0, H), rng.randint(0, W)
            sigma = rng.uniform(40, 120)
            amp = rng.uniform(0.2, 1.0)
            _add_gaussian(field, cy, cx, sigma, amp, H, W)
    else:
        raise ValueError(f"Unknown scene: {scene}")

    # Normalize to [0, 1]
    mn, mx = field.min(), field.max()
    if mx - mn > 1e-12:
        field = (field - mn) / (mx - mn)
    field = field.astype(np.float32)
    return field

# Quick test
gt_test = generate_gt("A_sparse_edges", 42, H, W)
print(f"GT shape: {gt_test.shape}, range: [{gt_test.min():.4f}, {gt_test.max():.4f}]")


In [ ]:
def coarse_from_gt(GT, downsample_factor):
    """Downsample then bilinear upsample — deliberately loses edges."""
    h, w = GT.shape
    small = zoom(GT, 1.0 / downsample_factor, order=1)
    coarse = zoom(small, downsample_factor, order=1)
    # Ensure exact shape match
    coarse = coarse[:h, :w]
    return coarse.astype(np.float32)

coarse_test = coarse_from_gt(gt_test, downsample_factor)
print(f"Coarse shape: {coarse_test.shape}, MSE vs GT: {np.mean((gt_test - coarse_test)**2):.6f}")


In [ ]:
def compute_tile_rho(GT, coarse, tile_size):
    """Compute rho = MSE per tile. Returns (tiles_y, tiles_x) array."""
    h, w = GT.shape
    ty = h // tile_size
    tx = w // tile_size
    rho = np.zeros((ty, tx), dtype=np.float32)
    for i in range(ty):
        for j in range(tx):
            y0, y1 = i * tile_size, (i + 1) * tile_size
            x0, x1 = j * tile_size, (j + 1) * tile_size
            diff = GT[y0:y1, x0:x1] - coarse[y0:y1, x0:x1]
            rho[i, j] = np.mean(diff ** 2)
    return rho

rho_test = compute_tile_rho(gt_test, coarse_test, tile_size)
print(f"Rho shape: {rho_test.shape}, max: {rho_test.max():.6f}, min: {rho_test.min():.6f}")


In [ ]:
def select_tiles_adaptive(rho, K):
    """Select top-K tiles by rho, tie-break by row-major index."""
    ty, tx = rho.shape
    # Build list of (rho_val, flat_index)
    entries = []
    for i in range(ty):
        for j in range(tx):
            flat_idx = i * tx + j
            entries.append((-rho[i, j], flat_idx))  # negative for ascending sort = descending rho
    entries.sort()
    selected = []
    for neg_rho, flat_idx in entries[:K]:
        i = flat_idx // tx
        j = flat_idx % tx
        selected.append((i, j))
    return selected

def select_tiles_random(rho, K, rng):
    """Select K random tiles."""
    ty, tx = rho.shape
    all_tiles = [(i, j) for i in range(ty) for j in range(tx)]
    indices = rng.choice(len(all_tiles), size=min(K, len(all_tiles)), replace=False)
    return [all_tiles[idx] for idx in sorted(indices)]

# Test
K_test = int(0.05 * n_tiles_total)
adaptive_sel = select_tiles_adaptive(rho_test, K_test)
print(f"K={K_test}, adaptive selected {len(adaptive_sel)} tiles")
print(f"Top-5 adaptive tiles: {adaptive_sel[:5]}")


In [ ]:
def refine(pred, GT, selected_tiles, tile_size, n_iter, alpha):
    """Iterative refinement: pred_tile += alpha * (GT_tile - pred_tile), n_iter times.
    Returns updated pred (copy). Cost = K * n_iter."""
    out = pred.copy()
    for _ in range(n_iter):
        for (i, j) in selected_tiles:
            y0, y1 = i * tile_size, (i + 1) * tile_size
            x0, x1 = j * tile_size, (j + 1) * tile_size
            out[y0:y1, x0:x1] += alpha * (GT[y0:y1, x0:x1] - out[y0:y1, x0:x1])
    return out

# Quick test
pred_a = refine(coarse_test.copy(), gt_test, adaptive_sel, tile_size, n_iter, alpha)
print(f"After adaptive refine: MSE = {np.mean((gt_test - pred_a)**2):.6f}")


In [ ]:
def compute_edge_mask(GT, edge_quantile, edge_eps):
    """Gradient-based edge mask with fallback."""
    dy = sobel(GT, axis=0)
    dx = sobel(GT, axis=1)
    grad = np.sqrt(dy**2 + dx**2)

    if grad.max() < edge_eps:
        return None, grad  # fallback: no edges

    threshold = np.quantile(grad, edge_quantile)
    edge_mask = grad > threshold
    return edge_mask, grad

def compute_metrics(GT, pred, edge_mask):
    """Compute MSE_global, PSNR, MSE_edge, MSE_non_edge."""
    mse_global = float(np.mean((GT - pred) ** 2))
    psnr = 10 * np.log10(1.0 / max(mse_global, 1e-12))

    if edge_mask is None:
        mse_edge = float('nan')
        mse_non_edge = mse_global
    else:
        diff2 = (GT - pred) ** 2
        if edge_mask.sum() > 0:
            mse_edge = float(np.mean(diff2[edge_mask]))
        else:
            mse_edge = float('nan')
        non_edge = ~edge_mask
        if non_edge.sum() > 0:
            mse_non_edge = float(np.mean(diff2[non_edge]))
        else:
            mse_non_edge = float('nan')

    return {
        "MSE_global": mse_global,
        "PSNR": psnr,
        "MSE_edge": mse_edge,
        "MSE_non_edge": mse_non_edge,
    }

# Test
edge_mask_test, grad_test = compute_edge_mask(gt_test, edge_quantile, edge_eps)
m = compute_metrics(gt_test, pred_a, edge_mask_test)
print(f"Metrics: {m}")
print(f"Edge pixels: {edge_mask_test.sum() if edge_mask_test is not None else 0}")


In [ ]:
scene_id_map = {"A_sparse_edges": 1, "B_many_edges": 2, "C_smooth_only": 3}

results = []

t_start = time.time()
for scene in scene_modes:
    s_id = scene_id_map[scene]
    for seed in range(n_seeds):
        GT = generate_gt(scene, seed, H, W)
        coarse = coarse_from_gt(GT, downsample_factor)
        rho = compute_tile_rho(GT, coarse, tile_size)
        edge_mask, _ = compute_edge_mask(GT, edge_quantile, edge_eps)

        # Coarse-only — coverage=0 reference point
        m_coarse = compute_metrics(GT, coarse, edge_mask)
        results.append({
            "scene": scene,
            "seed": seed,
            "coverage": 0.0,
            "K": 0,
            "budget": 0,
            "method": "coarse",
            **m_coarse,
        })

        for cov in coverage_list:
            K = max(1, int(cov * n_tiles_total))

            # ── Adaptive ──
            sel_adaptive = select_tiles_adaptive(rho, K)
            pred_adaptive = refine(coarse.copy(), GT, sel_adaptive, tile_size, n_iter, alpha)
            m_adaptive = compute_metrics(GT, pred_adaptive, edge_mask)

            # ── Baseline (random) ──
            # scene_id in RNG seed for cross-scene independence
            rng_baseline = np.random.RandomState(seed * 10000 + s_id * 100 + int(cov * 10000))
            sel_baseline = select_tiles_random(rho, K, rng_baseline)
            pred_baseline = refine(coarse.copy(), GT, sel_baseline, tile_size, n_iter, alpha)
            m_baseline = compute_metrics(GT, pred_baseline, edge_mask)

            budget = K * n_iter

            for method, m in [("adaptive", m_adaptive), ("baseline", m_baseline)]:
                results.append({
                    "scene": scene,
                    "seed": seed,
                    "coverage": cov,
                    "K": K,
                    "budget": budget,
                    "method": method,
                    **m,
                })

elapsed = time.time() - t_start
df = pd.DataFrame(results)
print(f"Experiment done in {elapsed:.1f}s — {len(df)} rows")
df.head(10)


In [ ]:
# Визуализация для первого seed, одной сцены, 2 coverages
viz_scene = scene_modes[0]
viz_seed = 0
viz_coverages = [0.05, 0.01]
viz_scene_id = {"A_sparse_edges": 1, "B_many_edges": 2, "C_smooth_only": 3}[viz_scene]

GT_viz = generate_gt(viz_scene, viz_seed, H, W)
coarse_viz = coarse_from_gt(GT_viz, downsample_factor)
rho_viz = compute_tile_rho(GT_viz, coarse_viz, tile_size)
edge_mask_viz, grad_viz = compute_edge_mask(GT_viz, edge_quantile, edge_eps)

img_count = 0
for cov in viz_coverages:
    if img_count >= max_static_images:
        break
    K = max(1, int(cov * n_tiles_total))

    sel_a = select_tiles_adaptive(rho_viz, K)
    sel_b = select_tiles_random(rho_viz, K,
                np.random.RandomState(viz_seed * 10000 + viz_scene_id * 100 + int(cov * 10000)))

    pred_a = refine(coarse_viz.copy(), GT_viz, sel_a, tile_size, n_iter, alpha)
    pred_b = refine(coarse_viz.copy(), GT_viz, sel_b, tile_size, n_iter, alpha)

    # Build masks for visualization
    mask_a = np.zeros((tiles_y, tiles_x), dtype=bool)
    for (i, j) in sel_a:
        mask_a[i, j] = True
    mask_b = np.zeros((tiles_y, tiles_x), dtype=bool)
    for (i, j) in sel_b:
        mask_b[i, j] = True

    fig, axes = plt.subplots(2, 5, figsize=(22, 9))
    fig.suptitle(f"{viz_scene} | seed={viz_seed} | coverage={cov} (K={K})", fontsize=14)

    panels = [
        ("GT", GT_viz),
        ("Coarse", coarse_viz),
        ("Baseline pred", pred_b),
        ("Adaptive pred", pred_a),
        ("|err| coarse", np.abs(GT_viz - coarse_viz)),
    ]
    for ax, (title, data) in zip(axes[0], panels):
        ax.imshow(data, cmap="viridis", vmin=0, vmax=1 if "err" not in title else None)
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    panels2 = [
        ("|err| baseline", np.abs(GT_viz - pred_b)),
        ("|err| adaptive", np.abs(GT_viz - pred_a)),
        ("rho heatmap", rho_viz),
        ("baseline mask", mask_b.astype(float)),
        ("adaptive mask", mask_a.astype(float)),
    ]
    for ax, (title, data) in zip(axes[1], panels2):
        cm = "hot" if "rho" in title or "mask" in title else "viridis"
        ax.imshow(data, cmap=cm)
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    img_count += 2  # count as 2 images (2 rows)

print(f"Static images rendered: {img_count}")


In [ ]:
# Графики: MSE, MSE_edge, PSNR vs coverage для каждой сцены
# coarse (coverage=0) shown as dashed horizontal reference line

for scene in scene_modes:
    sub = df[df["scene"] == scene]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    scene_label = scene
    if scene == "C_smooth_only":
        scene_label += "  (NB: MSE_edge = high-gradient zone, not hard edges)"
    fig.suptitle(f"Scene: {scene_label}", fontsize=13)

    for ax, metric in zip(axes, ["MSE_global", "MSE_edge", "PSNR"]):
        # Coarse reference line
        coarse_val = sub[sub["method"] == "coarse"][metric].mean()
        if not np.isnan(coarse_val):
            ax.axhline(coarse_val, color="gray", linestyle="--", linewidth=1, label="coarse (no refine)")

        for method, color, marker in [("adaptive", "tab:blue", "o"), ("baseline", "tab:orange", "s")]:
            grp = sub[(sub["method"] == method) & (sub["coverage"] > 0)].groupby("coverage")[metric]
            mean_ = grp.mean()
            std_ = grp.std()
            ax.errorbar(mean_.index, mean_.values, yerr=std_.values,
                        label=method, color=color, marker=marker, capsize=3)
        ax.set_xlabel("Coverage")
        ax.set_ylabel(metric)
        ax.set_title(metric)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


In [ ]:
# ═══════════════════════════════════════════
# Агрегирование и финальный вердикт
# ═══════════════════════════════════════════

# ── Coarse reference ──
print("=== COARSE REFERENCE (coverage=0) ===")
coarse_ref = df[df["method"] == "coarse"].groupby("scene")[["MSE_global", "MSE_edge", "PSNR"]].mean()
print(coarse_ref.round(6).to_string())
print()

# ── Adaptive vs Baseline comparison ──
summary_rows = []
for scene in scene_modes:
    for cov in coverage_list:
        sub = df[(df["scene"] == scene) & (df["coverage"] == cov)]
        a = sub[sub["method"] == "adaptive"]
        b = sub[sub["method"] == "baseline"]
        for metric in ["MSE_global", "MSE_edge", "PSNR"]:
            a_mean = a[metric].mean()
            b_mean = b[metric].mean()
            if metric == "PSNR":
                wins = a_mean > b_mean  # higher is better
                delta_pct = (a_mean - b_mean) / max(abs(b_mean), 1e-12) * 100
            else:
                wins = a_mean < b_mean  # lower is better
                delta_pct = (b_mean - a_mean) / max(abs(b_mean), 1e-12) * 100
            summary_rows.append({
                "scene": scene,
                "coverage": cov,
                "metric": metric,
                "adaptive_mean": round(a_mean, 7),
                "baseline_mean": round(b_mean, 7),
                "delta_%": round(delta_pct, 2),
                "adaptive_wins": wins,
            })

summary = pd.DataFrame(summary_rows)
print("=== ADAPTIVE vs BASELINE ===")
print(summary.to_string(index=False))

# Verdict
n_comparisons = len(summary)
n_wins = summary["adaptive_wins"].sum()
win_rate = n_wins / n_comparisons * 100

# Edge-specific
edge_summary = summary[summary["metric"] == "MSE_edge"].dropna(subset=["adaptive_mean"])
edge_wins = edge_summary["adaptive_wins"].sum()
edge_total = len(edge_summary)

print(f"\n{'='*60}")
print(f"Overall: adaptive wins {n_wins}/{n_comparisons} ({win_rate:.0f}%)")
print(f"MSE_edge: adaptive wins {edge_wins}/{edge_total}")
print(f"Note: for C_smooth_only, MSE_edge measures high-gradient zones (no hard edges)")

if win_rate >= 75 and (edge_total == 0 or edge_wins / max(edge_total, 1) >= 0.7):
    verdict = "✅ ГИПОТЕЗА ПОДТВЕРЖДЕНА — adaptive refinement работает"
else:
    verdict = "❌ ГИПОТЕЗА НЕ ПОДТВЕРЖДЕНА"

print(f"\n>>> {verdict}")
print(f"{'='*60}")


In [ ]:
df.to_csv("results.csv", index=False)
summary.to_csv("summary.csv", index=False)
print("Saved: results.csv, summary.csv")


In [ ]:
# ═══════════════════════════════════════════
# REPRO-CHECK
# ═══════════════════════════════════════════
print("Running repro-check...")
repro_scene = scene_modes[0]
repro_seed = 0
repro_cov = 0.05

for run in range(2):
    GT_r = generate_gt(repro_scene, repro_seed, H, W)
    coarse_r = coarse_from_gt(GT_r, downsample_factor)
    rho_r = compute_tile_rho(GT_r, coarse_r, tile_size)

    K_r = max(1, int(repro_cov * n_tiles_total))
    sel_r = select_tiles_adaptive(rho_r, K_r)
    pred_r = refine(coarse_r.copy(), GT_r, sel_r, tile_size, n_iter, alpha)
    edge_mask_r, _ = compute_edge_mask(GT_r, edge_quantile, edge_eps)
    m_r = compute_metrics(GT_r, pred_r, edge_mask_r)

    if run == 0:
        sel_first = sel_r
        m_first = m_r
    else:
        assert sel_r == sel_first, "REPRO FAIL: tile selection differs!"
        for key in m_first:
            v1, v2 = m_first[key], m_r[key]
            if np.isnan(v1) and np.isnan(v2):
                continue
            assert abs(v1 - v2) < 1e-10, f"REPRO FAIL: {key} differs: {v1} vs {v2}"

print("✅ Repro-check PASSED: tile selection and metrics are identical across runs.")
